# Stroke Prediction System

**Author:** Gauri Kotwal

An educational machine-learning project using patient demographic and health-related features to compare Logistic Regression and Random Forest classifiers.

> **Medical disclaimer:** This notebook is for learning and demonstration only. It is not a diagnostic tool and must not be used for clinical decisions.

## 1. Setup and imports

Install dependencies with `pip install -r requirements.txt` before running the notebook.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, roc_curve
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
DATA_PATH = Path("data/raw/healthcare-dataset-stroke-data.csv")
OUTPUT_DIR = Path("visualizations")
MODEL_DIR = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

## 2. Load the dataset

Download the CSV separately and place it at `data/raw/healthcare-dataset-stroke-data.csv`. The raw dataset is intentionally not stored in this repository.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. "
        "Download healthcare-dataset-stroke-data.csv and place it in data/raw/."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Initial data inspection

In [ ]:
display(df.info())
display(df.describe(include="all").T)
print("\nMissing values:")
display(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget distribution:")
display(df["stroke"].value_counts(dropna=False))

## 4. Data cleaning and exploratory analysis

The identifier column is retained in the exploratory dataframe but excluded from model features. Missing BMI values are summarized here; the modelling pipeline performs imputation using training data only to avoid preprocessing leakage.

In [ ]:
df = df.drop_duplicates().copy()

print("Missing BMI values:", df["bmi"].isna().sum())
print("Mean BMI (observed values):", round(df["bmi"].mean(), 2))

print("Records:", len(df))
print("Stroke cases:", int(df["stroke"].sum()))
print("Non-stroke cases:", int((df["stroke"] == 0).sum()))
print("Stroke percentage:", round(df["stroke"].mean() * 100, 2))

print("Average age:", round(df["age"].mean(), 2))
print("Average glucose:", round(df["avg_glucose_level"].mean(), 2))
print("Average BMI:", round(df["bmi"].mean(), 2))

In [ ]:
# Age distribution
plt.figure(figsize=(8, 5))
sns.histplot(df["age"], kde=True)
plt.title("Age Distribution")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "age_distribution.png", dpi=150)
plt.show()

# Target distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="stroke")
plt.title("Stroke Count (0 = No Stroke, 1 = Stroke)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "stroke_count.png", dpi=150)
plt.show()

In [ ]:
# Stroke distribution by selected categorical features
for column in ["gender", "smoking_status", "hypertension", "heart_disease"]:
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x=column, hue="stroke")
    plt.title(f"Stroke Distribution by {column.replace('_', ' ').title()}")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{column}_vs_stroke.png", dpi=150)
    plt.show()

In [ ]:
# Age groups for descriptive analysis only
df["Age_Group"] = pd.cut(
    df["age"],
    bins=[0, 30, 45, 60, 100],
    labels=["0-30", "31-45", "46-60", "61+"]
)

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Age_Group", hue="stroke")
plt.title("Stroke Distribution by Age Group")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "age_group_vs_stroke.png", dpi=150)
plt.show()

In [ ]:
# Numerical correlation heatmap
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=150)
plt.show()

display(df.groupby("stroke")[["age", "avg_glucose_level", "bmi"]].mean().round(2))

## 5. Prepare features and target

The rule-based `Risk_Level` label from early exploration is deliberately not used as a model feature or medical risk assessment. The target for supervised learning is the dataset's existing `stroke` column.

In [ ]:
ml_df = df.drop(columns=["id", "Age_Group"], errors="ignore").copy()
X = ml_df.drop(columns=["stroke"])
y = ml_df["stroke"].astype(int)

numerical_features = [
    "age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"
]
categorical_features = [
    "gender", "ever_married", "work_type", "Residence_type", "smoking_status"
]

print("Features:", X.shape)
print("Target:", y.shape)
display(y.value_counts())

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).rename("proportion"))
print("\nTesting target proportions:")
display(y_test.value_counts(normalize=True).rename("proportion"))

## 6. Train classification models

Both models use class weighting to account for the minority positive class. Class weighting does not remove the need for careful evaluation.

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_depth=10,
        min_samples_split=5
    ))
])

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)
print("Both models trained successfully.")

## 7. Evaluate models

Because the target is imbalanced, review recall, precision, F1-score and PR-AUC alongside accuracy. Precision indicates how many predicted positives were correct; recall indicates how many actual positives were detected.

In [ ]:
def evaluate_model(model, X_eval, y_eval, model_name):
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "F1 Score": f1_score(y_eval, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, probabilities),
        "PR-AUC": average_precision_score(y_eval, probabilities)
    }

    print(f"\n{model_name}")
    print(classification_report(y_eval, predictions, zero_division=0))
    return metrics, predictions, probabilities

lr_metrics, lr_pred, lr_prob = evaluate_model(
    logistic_model, X_test, y_test, "Logistic Regression"
)
rf_metrics, rf_pred, rf_prob = evaluate_model(
    random_forest_model, X_test, y_test, "Random Forest"
)

results = pd.DataFrame([lr_metrics, rf_metrics])
display(results.round(4))

In [ ]:
# Model metric comparison
plot_metrics = results.set_index("Model")[
    ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC", "PR-AUC"]
]
plot_metrics.plot(kind="bar", figsize=(12, 6))
plt.title("Machine Learning Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrices
for model_name, predictions, cmap, filename in [
    ("Logistic Regression", lr_pred, "Blues", "confusion_matrix_logistic_regression.png"),
    ("Random Forest", rf_pred, "Greens", "confusion_matrix_random_forest.png")
]:
    cm = confusion_matrix(y_test, predictions)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap=cmap,
        xticklabels=["No Stroke", "Stroke"],
        yticklabels=["No Stroke", "Stroke"]
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=150)
    plt.show()

In [ ]:
# ROC curve comparison
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)

plt.figure(figsize=(8, 6))
plt.plot(lr_fpr, lr_tpr, label="Logistic Regression")
plt.plot(rf_fpr, rf_tpr, label="Random Forest")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve_comparison.png", dpi=150)
plt.show()

## 8. Permutation feature importance

Permutation importance estimates how much the selected scoring metric changes when one feature is shuffled. It describes model behaviour on this test sample; it does not establish causation or medical importance.

In [ ]:
importance = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="average_precision"
)

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance.importances_mean
}).sort_values("Importance", ascending=False)

display(feature_importance.round(4))

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x="Importance", y="Feature")
plt.title("Permutation Feature Importance - Random Forest")
plt.xlabel("Mean decrease in average precision after permutation")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "permutation_feature_importance.png", dpi=150)
plt.show()

## 9. Save fitted models

The saved files contain the preprocessing steps and classifier together. They are generated locally and are excluded from Git by `.gitignore`.

In [ ]:
joblib.dump(logistic_model, MODEL_DIR / "stroke_logistic_model.joblib")
joblib.dump(random_forest_model, MODEL_DIR / "stroke_random_forest_model.joblib")

print("Model pipelines saved in:", MODEL_DIR.resolve())

## 10. Example prediction

This is an example prediction for one held-out dataset row. The probability is a model score, not an individual's clinical stroke risk.

In [ ]:
loaded_model = joblib.load(MODEL_DIR / "stroke_logistic_model.joblib")
sample_data = X_test.iloc[[0]]

prediction = loaded_model.predict(sample_data)[0]
probability = loaded_model.predict_proba(sample_data)[0, 1]

print("Predicted dataset class:", int(prediction))
print("Model-estimated probability for class 1:", round(float(probability), 4))

## 11. Project summary and interpretation

In the development run recorded in the repository README, Logistic Regression had higher recall, F1-score, ROC-AUC and PR-AUC, while Random Forest had higher accuracy. This illustrates why accuracy alone can be misleading on an imbalanced dataset.

The models are exploratory baselines. Stronger validation, threshold selection on validation data, calibration, fairness analysis and external validation would be needed even before considering any clinical research use.

**This project is for education only and is not suitable for medical diagnosis or treatment decisions.**